# NetCDF to DGGS (H3) Conversion

This notebook provides an optimized approach for converting NetCDF climate data to multi-resolution DGGS H3 format.

## Overview

This notebook the following operations:
- Processes NetCDF climate data directly to Zarr format using `xarray` operations
- Generates multi-resolution DGGS data (resolution 0 to optimal base resolution)
- Creates proper DataTree structure with resolution groups for [`pydggsapi`](https://github.com/LandscapeGeoinformatics/pydggsapi)
- Generates complete `pydggsapi` configuration with relevant spatiotemporal collection metadata
- Extracts climate variables metadata from NetCDF attributes and [ClimateData.ca](https://climatedata.ca/variables/)

## What This Notebook Does

### 1. Timing Infrastructure
- Tracks individual operation times:
  - NetCDF open
  - H3 grid computation/loading
  - Coordinate assignment
  - Stack operation
  - Drop NaN
  - Groupby aggregation
  - Zarr write
  - Total time

### 2. Intermediate Results & Progress Tracking

- Processes all NetCDF files for specified climate indices
- Skips already-processed files (resume capability)
- Collects timing statistics per file
- Saves comprehensive JSON summary:
  ```json
  {
    "h3_resolution": 7,
    "total_files": 12,
    "processed_files": 12,
    "total_time_seconds": 245.3,
    "timings": [...],
    "timestamp": "2026-03-10T..."
  }
  ```

### 3. Full Monthly Processing

**Configuration**:
- `CLIMATE_INDICES = ["prcptot", "tx_max", "ice_days", "frost_days"]`
- `AGG_TYPES = ["MS"]` - Monthly (MS) or Yearly (YS) aggregation types
- Finds all NetCDF files in `data/allyears/{indice}/{agg_type}/`
- Automatically determines H3 resolution from first 10 files
- Processes ALL files for ALL indices and aggregation types

**Outputs**:
- Individual Zarr stores per file: `{filename}_h3_res={resolution}.zarr`
- Timing summary JSON: `timing_summary_res={resolution}.json`

### 4. Parent Resolution Aggregation

- Processes resolutions on cascading aggregation: H3_RESOLUTION → ... → 6 → 5 → 4 → 3 → 2 → 1 → 0
- Organizes output in subdirectories: `outputs/dggs/h3/res={N}/`
- Saves summary JSON with per-resolution timing

### Output Structure

After processing, your output directory will contain:
```
outputs/
├── dggs/
│   └── h3/
│       ├── res=0/                               # Coarsest resolution
│       │   └── {file}_h3_res=0.zarr
│       ├── res=1/
│       │   └── {file}_h3_res=1.zarr
│       ├── ...
│       ├── res=6/                               # Base resolution (finest)
│       │   └── {file}_h3_res=6.zarr
│       └── canada_climate_h3_dggs.zarr/         # Final combined hierarchical zarr
│           ├── res0/                            # Group for resolution 0
│           ├── res1/                            # Group for resolution 1
│           ├── ...
│           └── res6/                            # Group for resolution 6
└── pydggsapi_config.json                        # pydggsapi configuration
```

### Cache Management

To reprocess data:
- Clear H3 grid cache: Remove files in `{CACHE_DIR}/h3_grid_*.npy`
- Clear metadata cache: Remove `{CACHE_DIR}/climate_variable_metadata.json`
- Clear processed zarr files: Remove directories in `{OUTPUT_DIR}/dggs/h3/res=*/`
- Clear final combined zarr: Remove `{FINAL_ZARR_PATH}`

---

## Technical Approach

**⚠️ Important Consideration ⚠️**

Both NetCDF and Zarr are chunked array formats. We leverage `xarray` to skip intermediate conversions with other formats.
However, we assume that the NetCDF files we are interested in are mostly aligned spatiotemporally already to skip certain steps.
For example, chucking will take into consideration that the time dimension is consistent across all files,
so the same chunking strategy applies identically for all DGGS resolutions to avoid fragmentation.

**Strategy**:
1. Compute `(lat, lon) → h3_id` mapping once per grid
2. Add H3 as coordinate to `xarray.Dataset`
3. Use `xarray`'s `groupby` to aggregate (time, lat, lon) → (time, h3_id)
4. Write directly to Zarr with optimal chunking

### Important Assumptions & Design Decisions

**Spatiotemporal Alignment**:
- NetCDF files are assumed to be mostly aligned spatiotemporally
- This allows skipping redundant alignment/resampling steps
- All files within an index/aggregation type share the same temporal grid

**Chunking Strategy**:
- **Spatial dimension (`h3_id`)**: Single chunk containing all H3 cells
  - Minimizes chunk overhead (metadata, compression headers)
  - Optimal for spatial queries (common in DGGS use cases)
  - Avoids fragmentation from `groupby()` operations

- **Temporal dimension (`time`)**: Single chunk containing all timesteps
  - Maintains consistency across all DGGS resolutions
  - Parent DGGS resolutions inherit temporal chunking from base resolution
  - Ensures uniform query performance across resolution hierarchy

**Why Chunking Matters**:
- Without explicit chunking, `groupby()` creates fragmented chunks (1000s of small chunks)
- Fragmentation causes massive storage bloat (easily 10x more storage for the same data) and poor read performance
- Small chunks have poor compression ratios and high metadata overhead
- Consistent chunking across resolutions enables predictable query performance


## Package Setup

### Installation
Only run this cell if packages are missing. Most should be available in the ogc-dggs environment.

In [ ]:
!pip install xarray zarr h3 numpy pandas tqdm beautifulsoup4 requests netcdf4


### Imports

In [28]:
import datetime
from pathlib import Path
from tqdm.auto import tqdm
import h3
import json
import numpy as np
import os
import pandas as pd
import requests
import time
import warnings
import xarray as xr
import zarr
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", message="Consolidated metadata is currently not part in the Zarr format 3 specification")

## Configuration

Define all paths and processing parameters.


In [31]:
def ensure_directory(path):
    """
    Ensure directory exists, creating it or following symlinks as needed.
    """
    path = Path(path)

    if path.is_symlink():
        target = path.resolve()
        target.mkdir(parents=True, exist_ok=True)
        return target
    else:
        path.mkdir(parents=True, exist_ok=True)
        return path


In [ ]:
# Climate indices to process
CLIMATE_INDICES = ["prcptot", "tx_max"]  # ["prcptot", "tx_max", "ice_days", "frost_days"]

# Aggregation types: Monthly (MS) or Yearly (YS)
AGG_TYPES = ["MS"]

# Directory paths
BASE_DIR = Path(os.getenv("CLIMATE_DATA_BASE_DIR", "data/allyears"))
CACHE_DIR = ensure_directory(os.getenv("CLIMATE_CACHE_DIR", "cache"))
OUTPUT_DIR = ensure_directory(os.getenv("CLIMATE_OUTPUT_DIR", "outputs"))

# DGGS output hierarchy (consistent structure for all zarr outputs)
DGGS_OUTPUT_DIR = OUTPUT_DIR / "dggs"
H3_OUTPUT_DIR = DGGS_OUTPUT_DIR / "h3"

# Final output paths
FINAL_ZARR_PATH = H3_OUTPUT_DIR / "canada_climate_h3_dggs.zarr"
FINAL_CONFIG_PATH = OUTPUT_DIR / "pydggsapi_config.json"

print(f"Configuration loaded:")
print(f"  Climate indices: {CLIMATE_INDICES}")
print(f"  Aggregation types: {AGG_TYPES}")
print(f"  Base directory: {BASE_DIR}")
print(f"  Cache directory: {CACHE_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  DGGS output hierarchy: {H3_OUTPUT_DIR}")
print(f"  Final zarr output: {FINAL_ZARR_PATH}")
print(f"  Final config output: {FINAL_CONFIG_PATH}")


## Step 1: Determine Optimal H3 Resolution

Analyze NetCDF grid spacing and find the best matching H3 resolution.


In [20]:
def analyze_netcdf_resolution(nc_files):
    """
    Analyze spatial resolution of NetCDF files and determine optimal H3 resolution.

    Returns:
        tuple: (h3_resolution, grid_info_dict)
    """
    print(f"Analyzing {len(nc_files)} NetCDF files for spatial resolution...")

    resolutions = []
    for f in tqdm(nc_files, desc="Analyzing files"):
        ds = xr.open_dataset(f, decode_timedelta=False)
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]

        # Estimate resolution (in degrees)
        lat_res = float(np.abs(lat[1] - lat[0]))
        lon_res = float(np.abs(lon[1] - lon[0]))

        # Approximate km using haversine formula
        mean_lat = float(lat.mean())
        earth_radius_km = 6371.0
        lat_km = lat_res * (np.pi/180) * earth_radius_km
        lon_km = lon_res * (np.pi/180) * earth_radius_km * np.cos(mean_lat * np.pi/180)

        resolutions.append((f, lat_km, lon_km))
        ds.close()

    # Find best (finest) resolution
    best_file, best_lat_km, best_lon_km = min(resolutions, key=lambda x: min(x[1], x[2]))
    print(f"\nBest spatial resolution: {best_lat_km:.2f} km x {best_lon_km:.2f} km")
    print(f"  from: {Path(best_file).name}")

    # Map NetCDF Grid to H3 DGGS
    # Find the finest H3 resolution where edge <= grid resolution
    h3_resolution = None
    for res in range(16):
        h3_edge_km = h3.average_hexagon_edge_length(res, "km")
        if h3_edge_km <= min(best_lat_km, best_lon_km):
            h3_resolution = res
            break

    if h3_resolution is None:
        h3_resolution = 15
        h3_edge_km = h3.average_hexagon_edge_length(h3_resolution, "km")
    else:
        h3_edge_km = h3.average_hexagon_edge_length(h3_resolution, "km")

    print(f"\nSelected H3 resolution: {h3_resolution} (edge ~{h3_edge_km:.3f} km)")
    print(f"  Reasoning: H3 res {h3_resolution} has edge {h3_edge_km:.3f} km <= grid {min(best_lat_km, best_lon_km):.2f} km")

    return h3_resolution, {
        'best_lat_km': best_lat_km,
        'best_lon_km': best_lon_km,
        'h3_edge_km': h3_edge_km,
        'best_file': best_file
    }


## Step 2: Helper Functions

### Output Path Generation

Ensure consistent directory structure across all functions.


In [33]:
def get_dggs_output_path(h3_resolution, filename=None):
    """
    Generate consistent output path for DGGS H3 zarr files.
    """
    output_dir = H3_OUTPUT_DIR / f"res={h3_resolution}"

    if filename:
        return output_dir / filename
    return output_dir


### Vectorized H3 Grid Computation

Compute H3 indices for entire lat/lon grid at once (no loops!)


In [34]:
def compute_h3_grid(lat, lon, resolution):
    """
    Compute H3 indices for a lat/lon grid (vectorized).
    Returns 2D array of H3 indices as int64.
    """
    lat_grid, lon_grid = np.meshgrid(lat, lon, indexing='ij')
    lat_flat = lat_grid.ravel()
    lon_flat = lon_grid.ravel()

    # Still need list comprehension for h3 library, but vectorized the grid creation
    h3_cells = np.array([
        int(h3.latlng_to_cell(float(la), float(lo), resolution), 16)
        for la, lo in zip(lat_flat, lon_flat)
    ], dtype=np.int64)

    return h3_cells.reshape(lat_grid.shape)


## Step 3: Process Single NetCDF File to H3 Zarr

Process one NetCDF file, all variables at once, single resolution.


In [35]:
def process_netcdf_to_h3_single_resolution(nc_file, h3_resolution, skip_existing=True):
    """
    Process NetCDF to H3 DGGS at single resolution using xarray operations.
    """
    timings = {}
    start_total = time.time()

    print(f"\nProcessing: {Path(nc_file).name}")

    # Create resolution-specific output directory using helper function
    output_dir = get_dggs_output_path(h3_resolution)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_zarr = output_dir / f"{Path(nc_file).stem}_h3_res={h3_resolution}.zarr"

    # Check if already exists
    if skip_existing and output_zarr.exists():
        print(f"  ⏭️  Output already exists, skipping")
        ds_existing = xr.open_zarr(output_zarr, decode_timedelta=False)
        timings['total'] = 0
        timings['skipped'] = True
        return ds_existing, timings, output_zarr

    # Open NetCDF (no chunking yet, keep it simple)
    t0 = time.time()
    ds = xr.open_dataset(nc_file, decode_timedelta=False)
    timings['open_netcdf'] = time.time() - t0

    # ...existing code...
    lat = ds['lat'].values if 'lat' in ds else ds['latitude'].values
    lon = ds['lon'].values if 'lon' in ds else ds['longitude'].values

    # Compute or load cached H3 grid mapping
    grid_hash = hash((tuple(lat), tuple(lon), h3_resolution))
    cache_file = Path(CACHE_DIR) / f"h3_grid_{grid_hash}.npy"

    t0 = time.time()
    if cache_file and cache_file.exists():
        print(f"  Loading cached H3 grid...")
        h3_grid = np.load(cache_file)
    else:
        print(f"  Computing H3 grid (res={h3_resolution})...")
        h3_grid = compute_h3_grid(lat, lon, h3_resolution)
        if cache_file:
            cache_file.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_file, h3_grid)
    timings['h3_grid'] = time.time() - t0

    # Add H3 index as a coordinate
    print(f"  Adding H3 coordinate to dataset...")
    t0 = time.time()
    ds = ds.assign_coords({'h3_index': (('lat', 'lon'), h3_grid)})
    timings['assign_coords'] = time.time() - t0

    # Stack spatial dimensions: (time, lat, lon) → (time, spatial)
    print(f"  Stacking spatial dimensions...")
    t0 = time.time()
    ds_stacked = ds.stack(spatial=['lat', 'lon'])
    timings['stack'] = time.time() - t0

    # Drop NaN values (ocean/missing data)
    print(f"  Dropping NaN values...")
    t0 = time.time()
    ds_stacked = ds_stacked.dropna(dim='spatial', how='all')
    timings['dropna'] = time.time() - t0

    # Group by H3 and aggregate: (time, spatial) → (time, h3_id)
    print(f"  Aggregating by H3 cell...")
    t0 = time.time()
    ds_h3 = ds_stacked.groupby('h3_index').mean()
    timings['groupby_aggregate'] = time.time() - t0

    # Rename dimension for clarity
    ds_h3 = ds_h3.rename({'h3_index': 'h3_id'})

    # Rechunk for efficient storage
    # Spatial: all h3 cells in one chunk for minimal overhead
    # Temporal: all timesteps in one chunk to maintain consistency across resolutions
    optimal_chunks = {
        'h3_id': -1,  # Single chunk for all h3 cells
        'time': ds_h3.sizes['time'] if 'time' in ds_h3.sizes else -1
    }
    ds_h3 = ds_h3.chunk(optimal_chunks)

    # Write to Zarr
    print(f"  Writing to Zarr: {output_zarr}")
    t0 = time.time()
    ds_h3.to_zarr(output_zarr, mode='w')
    timings['write_zarr'] = time.time() - t0

    ds.close()

    timings['total'] = time.time() - start_total

    print(f"  ⏱️  Total time: {timings['total']:.2f}s")
    print(f"     ├─ H3 grid: {timings['h3_grid']:.2f}s")
    print(f"     ├─ Stack/aggregate: {timings['stack'] + timings['groupby_aggregate']:.2f}s")
    print(f"     └─ Write Zarr: {timings['write_zarr']:.2f}s")

    return ds_h3, timings, output_zarr


## Step 4: Test with Actual Data

Determine H3 resolution from data, then test processing.


In [ ]:

# Find test files
test_files = list(BASE_DIR.glob("**/*prcptot*.nc"))
if test_files:
    print(f"Found {len(test_files)} prcptot files")

    # Determine optimal H3 resolution from the data
    H3_RESOLUTION, grid_info = analyze_netcdf_resolution(test_files[:5])  # Analyze first 5 files

    print(f"\n{'='*60}")
    print(f"Will use H3 resolution: {H3_RESOLUTION}")
    print(f"{'='*60}\n")

    # Process first file as test
    test_file = test_files[0]
    print(f"Test file: {test_file}")

    # Process it (will skip if already exists)
    result, timings, output_path = process_netcdf_to_h3_single_resolution(
        test_file,
        H3_RESOLUTION,
        skip_existing=True
    )

    print(f"\n✅ Test completed successfully!")
    print(f"Result shape: {result.dims}")
    print(f"Variables: {list(result.data_vars)}")
    print(f"Output: {output_path}")
else:
    print("No test files found")


## Step 5: Process All Monthly NetCDF Files

Now process all months for all climate indices.


In [37]:
def process_all_nc_files(nc_files, h3_resolution, cache_dir):
    """
    Process all NetCDF files.

    Args:
        nc_files: List of NetCDF file paths to process
        h3_resolution: H3 resolution to use
        cache_dir: Cache directory

    Returns:
        dict: Summary statistics and timing info
    """
    all_timings = []
    processed_files = []

    all_nc_files = sorted(set(nc_files))  # Remove duplicates

    print(f"\n{'='*70}")
    print(f"Processing {len(all_nc_files)} NetCDF files at H3 resolution {h3_resolution}")
    print(f"{'='*70}\n")

    start_total = time.time()

    for i, nc_file in enumerate(all_nc_files, 1):
        print(f"\n[{i}/{len(all_nc_files)}] {nc_file.name}")

        try:
            result, timings, output_path = process_netcdf_to_h3_single_resolution(
                nc_file,
                h3_resolution,
                skip_existing=True
            )

            # Only add timing if actually processed (not skipped)
            if not timings.get('skipped', False):
                all_timings.append({
                    'file': nc_file.name,
                    'variables': list(result.data_vars),
                    'n_variables': len(result.data_vars),
                    'n_cells': result.sizes.get('h3_id', 0),
                    'n_timesteps': result.sizes.get('time', 0),
                    **timings
                })

            processed_files.append(output_path)

        except Exception as e:
            print(f"  ❌ Error processing {nc_file.name}: {e}")
            continue

    total_time = time.time() - start_total

    print(f"\n{'='*70}")
    print(f"✅ Processing complete!")
    print(f"{'='*70}")
    print(f"Total time: {total_time/60:.2f} minutes ({total_time:.1f}s)")
    print(f"Processed: {len(processed_files)} files")
    print(f"Average time per file: {total_time/max(len(all_timings), 1):.2f}s")

    # Save timing summary
    timing_summary = {
        'h3_resolution': h3_resolution,
        'total_files': len(all_nc_files),
        'processed_files': len(processed_files),
        'total_time_seconds': total_time,
        'timings': all_timings,
        'timestamp': datetime.datetime.now().isoformat()
    }

    summary_file = Path(OUTPUT_DIR) / f"timing_summary_res={h3_resolution}.json"
    with open(summary_file, 'w') as f:
        json.dump(timing_summary, f, indent=2)

    print(f"Timing summary saved to: {summary_file}")

    return timing_summary


## Step 6: Execute Full Processing

Process all monthly files for all climate indices.


In [25]:
# Find all files matching indices and aggregation types
all_files = []
for idx in CLIMATE_INDICES:
    for agg_type in AGG_TYPES:
        # Pattern: data/allyears/{indice}/{agg_type}/*.nc
        pattern = BASE_DIR / idx / agg_type / "*.nc"
        files = list(pattern.parent.glob("*.nc")) if pattern.parent.exists() else []
        all_files.extend(files)
all_files = sorted(set(all_files))

if all_files:
    # Determine H3 resolution by sampling first 10 files
    sample_size = min(10, len(all_files))
    print(f"\nDetermining optimal H3 resolution from {sample_size} sample file(s)...\n")
    H3_RESOLUTION, grid_info = analyze_netcdf_resolution(all_files[:sample_size])

    print(f"\n{'='*70}")
    print(f"CONFIGURATION")
    print(f"{'='*70}")
    print(f"Climate indices: {CLIMATE_INDICES}")
    print(f"Aggregation types: {AGG_TYPES}")
    print(f"H3 resolution: {H3_RESOLUTION}")
    print(f"Total files to process: {len(all_files)}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

    # Process all files
    summary = process_all_nc_files(
        nc_files=all_files,
        h3_resolution=H3_RESOLUTION,
        cache_dir=CACHE_DIR
    )
else:
    print("No NetCDF files found!")



Determining optimal H3 resolution from 10 sample file(s)...

Analyzing 10 NetCDF files for spatial resolution...


Analyzing files:   0%|          | 0/10 [00:00<?, ?it/s]


Best spatial resolution: 9.27 km x 4.31 km
  from: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Selected H3 resolution: 6 (edge ~3.725 km)
  Reasoning: H3 res 6 has edge 3.725 km <= grid 4.31 km

CONFIGURATION
Climate indices: ['prcptot', 'tx_max']
Aggregation types: ['MS']
H3 resolution: 6
Total files to process: 24
Output directory: /misc/scratch13/OGC11/canada-climate/outputs


Processing 24 NetCDF files at H3 resolution 6


[1/24] prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc
  ⏭️  Output already exists, skipping

[2/24] prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc
  ⏭️  Output already exists, skipping

[3/24] prcptot_MS_MBCn+PCIC-Blend_histori

## Step 7: Aggregate to Parent DGGS Resolutions

After processing at the finest resolution, aggregate to all parent resolutions.


In [26]:
def aggregate_to_parent_resolution(source_zarr, parent_res, output_zarr):
    """
    Aggregate H3 data from higher resolution to a parent resolution.

    Args:
        source_zarr: Path to source Zarr store (higher resolution)
        parent_res: Target parent resolution (lower number = coarser)
        output_zarr: Output Zarr path

    Returns:
        xarray Dataset at parent resolution
    """
    print(f"  Aggregating to parent resolution {parent_res}...")

    # Open source data
    ds = xr.open_zarr(source_zarr, decode_timedelta=False)

    # Get current H3 IDs (int64)
    h3_ids = ds['h3_id'].values

    # Convert to parent H3 IDs
    print(f"    Converting {len(h3_ids)} cells to parent res...")
    parent_h3_ids = np.array([
        int(h3.cell_to_parent(h3.int_to_str(h3_id), parent_res), 16)
        for h3_id in h3_ids
    ], dtype=np.int64)

    # Add parent H3 ID as coordinate
    ds = ds.assign_coords({'h3_parent': ('h3_id', parent_h3_ids)})

    # Group by parent and aggregate
    print(f"    Grouping and aggregating...")
    ds_parent = ds.groupby('h3_parent').mean()
    ds_parent = ds_parent.rename({'h3_parent': 'h3_id'})

    # Rechunk for efficient storage, preserving temporal chunking from source
    # This ensures consistency across all resolutions
    source_time_chunks = ds.chunks.get('time', None) if hasattr(ds, 'chunks') else None

    optimal_chunks = {
        'h3_id': -1,  # Single chunk for all h3 cells
    }

    # Preserve time chunking from source if available, otherwise use all timesteps
    if 'time' in ds_parent.sizes:
        if source_time_chunks and len(source_time_chunks) > 0:
            # Use same time chunking as source
            optimal_chunks['time'] = source_time_chunks[0]
        else:
            # Fallback: all timesteps in one chunk
            optimal_chunks['time'] = ds_parent.sizes['time']

    ds_parent = ds_parent.chunk(optimal_chunks)

    # Write to Zarr
    print(f"    Writing to {output_zarr}")
    ds_parent.to_zarr(output_zarr, mode='w')

    ds.close()

    return ds_parent

def process_all_parent_resolutions(base_resolution):
    """
    Process all parent resolutions from base_resolution down to 0.

    Args:
        base_resolution: Starting (finest) H3 resolution

    Returns:
        dict: Summary of processed resolutions
    """
    # Find source files using helper function
    source_path = get_dggs_output_path(base_resolution)
    source_files = sorted(source_path.glob(f"*_h3_res={base_resolution}.zarr"))

    print(f"\n{'='*70}")
    print(f"Aggregating {len(source_files)} variables to parent resolutions")
    print(f"Base resolution: {base_resolution} → Target: 0-{base_resolution-1}")
    print(f"{'='*70}\n")

    summary = {}

    for parent_res in range(base_resolution - 1, -1, -1):
        print(f"\n{'='*70}")
        print(f"Processing resolution {parent_res}")
        print(f"{'='*70}")

        res_start = time.time()
        # Use helper function for consistent output path
        output_dir = get_dggs_output_path(parent_res)
        output_dir.mkdir(parents=True, exist_ok=True)

        processed_count = 0

        for source_file in tqdm(source_files, desc=f"Res {parent_res}"):
            # Determine source for this resolution
            if parent_res == base_resolution - 1:
                # First parent level: aggregate from base resolution
                source_zarr = source_file
            else:
                # Subsequent levels: aggregate from previous parent using helper function
                prev_dir = get_dggs_output_path(parent_res + 1)
                source_zarr = prev_dir / source_file.name.replace(
                    f"_h3_res={base_resolution}.zarr",
                    f"_h3_res={parent_res + 1}.zarr"
                )

                if not source_zarr.exists():
                    print(f"Warning: Source not found: {source_zarr}")
                    continue

            # Output file for this parent resolution
            output_zarr = output_dir / source_file.name.replace(
                f"_h3_res={base_resolution}.zarr",
                f"_h3_res={parent_res}.zarr"
            )

            if output_zarr.exists():
                continue

            try:
                aggregate_to_parent_resolution(source_zarr, parent_res, output_zarr)
                processed_count += 1
            except Exception as e:
                print(f"Error processing {source_file.name} to res {parent_res}: {e}")
                continue

        res_time = time.time() - res_start
        summary[parent_res] = {
            'processed': processed_count,
            'time_seconds': res_time
        }

        print(f"\n✅ Resolution {parent_res} complete: {processed_count} variables in {res_time:.1f}s")

    return summary


## Step 8: Execute Parent Resolution Aggregation

Aggregate all processed data to parent resolutions.


In [27]:
if 'H3_RESOLUTION' in globals() and H3_RESOLUTION is not None:
    print(f"Starting parent resolution aggregation from resolution {H3_RESOLUTION}")

    parent_summary = process_all_parent_resolutions(base_resolution=H3_RESOLUTION)

    print(f"\n{'='*70}")
    print(f"✅ All parent resolutions complete!")
    print(f"{'='*70}")

    for res, info in sorted(parent_summary.items(), reverse=True):
        print(f"Resolution {res}: {info['processed']} variables in {info['time_seconds']:.1f}s")

    # Save summary
    summary_file = OUTPUT_DIR / "parent_resolutions_summary.json"
    with open(summary_file, mode="w", encoding="utf-8") as f:
        json.dump(parent_summary, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
else:
    print("H3_RESOLUTION not defined. Run previous cells first.")

Starting parent resolution aggregation from resolution 6

Aggregating 25 variables to parent resolutions
Base resolution: 6 → Target: 0-5


Processing resolution 5


Res 5:   0%|          | 0/25 [00:00<?, ?it/s]


✅ Resolution 5 complete: 0 variables in 0.0s

Processing resolution 4


Res 4:   0%|          | 0/25 [00:00<?, ?it/s]

  Aggregating to parent resolution 4...
    Converting 45661 cells to parent res...
    Grouping and aggregating...



KeyboardInterrupt



## Next: Process All Variables in One File

Since all variables share the same grid, we can process them together.


In [ ]:
def process_file_all_variables(nc_file, h3_resolution, output_base_dir, cache_dir=None,
                                climate_indices=None):
    """
    Process all matching climate variables in one NetCDF file.

    Args:
        nc_file: Path to NetCDF file
        h3_resolution: H3 resolution level
        output_base_dir: Base directory for output Zarr stores
        cache_dir: Optional cache directory for H3 grid
        climate_indices: List of climate index names to filter (e.g., ['prcptot', 'tx_max'])

    Returns:
        Dict mapping variable names to output Zarr paths
    """
    print(f"\nProcessing all variables in: {Path(nc_file).name}")

    ds = xr.open_dataset(nc_file, decode_timedelta=False)

    # Find matching variables
    if climate_indices:
        matching_vars = [v for v in ds.data_vars
                        if any(idx in v for idx in climate_indices)]
    else:
        matching_vars = list(ds.data_vars)

    print(f"  Found {len(matching_vars)} matching variables")

    if not matching_vars:
        ds.close()
        return {}

    # Get coordinates
    lat = ds['lat'].values if 'lat' in ds else ds['latitude'].values
    lon = ds['lon'].values if 'lon' in ds else ds['longitude'].values

    # Compute H3 grid (once for all variables)
    grid_hash = hash((tuple(lat), tuple(lon), h3_resolution))
    cache_file = Path(cache_dir) / f"h3_grid_{grid_hash}.npy" if cache_dir else None

    if cache_file and cache_file.exists():
        print(f"  Loading cached H3 grid...")
        h3_grid = np.load(cache_file)
    else:
        print(f"  Computing H3 grid (res={h3_resolution})...")
        h3_grid = compute_h3_grid(lat, lon, h3_resolution)
        if cache_file:
            cache_file.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_file, h3_grid)

    # Process each variable
    output_paths = {}
    for var_name in tqdm(matching_vars, desc="  Variables"):
        # Create dataset with just this variable
        ds_var = ds[[var_name]]
        ds_var = ds_var.assign_coords({'h3_index': (('lat', 'lon'), h3_grid)})

        # Stack and aggregate
        ds_stacked = ds_var.stack(spatial=['lat', 'lon'])
        ds_stacked = ds_stacked.dropna(dim='spatial', how='all')
        ds_h3 = ds_stacked.groupby('h3_index').mean()
        ds_h3 = ds_h3.rename({'h3_index': 'h3_id'})

        # Write to Zarr
        output_zarr = Path(output_base_dir) / f"{var_name}_h3_res={h3_resolution}.zarr"
        ds_h3.to_zarr(output_zarr, mode='w')
        output_paths[var_name] = output_zarr

    ds.close()
    return output_paths



## Step 9: Combine Individual Zarr Files into Hierarchical Zarr Structure

pydggsapi expects a hierarchical zarr structure with resolution-based groups.
We need to combine all individual zarr files into a single zarr store with groups like:
- `/res0/` - Resolution 0 data
- `/res1/` - Resolution 1 data
- ...
- `/res6/` - Resolution 6 data

We use `xarray`'s built-in `group` parameter to create this hierarchical structure.


In [ ]:
def combine_zarr_files_to_grouped_zarr(output_base_dir, h3_resolution, final_output_path):
    """
    Combine individual zarr files across all resolutions into a single hierarchical zarr
    with resolution-based groups for pydggsapi.

    Args:
        output_base_dir: Base directory containing dggs/h3/res={N}/ subdirectories
        h3_resolution: Maximum H3 resolution (base resolution)
        final_output_path: Path for final combined zarr store

    Returns:
        dict: Mapping of resolution to group names
    """
    print(f"\n{'='*70}")
    print(f"Combining zarr files into hierarchical zarr structure")
    print(f"{'='*70}\n")

    final_output_path = Path(final_output_path)
    base_path = Path(output_base_dir) / "dggs" / "h3"

    # Remove existing output if it exists
    if final_output_path.exists():
        import shutil
        shutil.rmtree(final_output_path)

    group_mapping = {}

    # Process each resolution from 0 to h3_resolution
    for res in range(h3_resolution + 1):
        res_dir = base_path / f"res={res}"

        if not res_dir.exists():
            print(f"⚠️  Resolution {res} directory not found: {res_dir}")
            continue

        zarr_files = sorted(res_dir.glob("*.zarr"))

        if not zarr_files:
            print(f"⚠️  No zarr files found for resolution {res}")
            continue

        print(f"\nProcessing resolution {res}: {len(zarr_files)} file(s)")

        # Open all zarr files for this resolution
        datasets = []
        for zarr_file in tqdm(zarr_files, desc=f"  Loading res {res}"):
            try:
                ds = xr.open_zarr(zarr_file, decode_timedelta=False)
                datasets.append(ds)
            except Exception as e:
                print(f"    ⚠️  Failed to open {zarr_file.name}: {e}")
                continue

        if not datasets:
            print(f"  ⚠️  No valid datasets for resolution {res}")
            continue

        # Combine all datasets for this resolution
        print(f"  Merging {len(datasets)} dataset(s)...")

        if len(datasets) == 1:
            combined_ds = datasets[0]
        else:
            # Merge datasets - they should have compatible h3_id and time coordinates
            # Chunking from individual datasets is preserved during merge
            combined_ds = xr.merge(datasets, compat='override')

        # Save to zarr group
        group_name = f"res{res}"
        print(f"  Writing to group '{group_name}'...")
        print(f"    Variables: {len(combined_ds.data_vars)}")
        print(f"    H3 cells: {combined_ds.sizes.get('h3_id', 0):,}")
        print(f"    Timesteps: {combined_ds.sizes.get('time', 0)}")

        # Write to zarr using the group parameter
        # Chunking is inherited from the merged datasets (already optimized)
        # Use consolidated=False here because we're writing multiple groups incrementally
        # Consolidation happens once at the end after ALL resolution groups are written
        # This is more efficient than re-consolidating after each group write
        mode = 'w' if res == 0 else 'a'
        combined_ds.to_zarr(final_output_path, group=group_name, mode=mode, consolidated=False)

        group_mapping[str(res)] = group_name

        # Close individual datasets
        for ds in datasets:
            ds.close()
        combined_ds.close()

    # Consolidate metadata once at the end for all groups
    # This creates a single .zmetadata file with metadata for all resolution groups
    # Making subsequent reads much faster (single metadata read instead of per-group)
    print(f"\n{'='*70}")
    print(f"Consolidating metadata...")
    print(f"{'='*70}\n")
    zarr.consolidate_metadata(final_output_path)

    print(f"✅ Hierarchical zarr created with {len(group_mapping)} resolution group(s)")
    print(f"   Groups: {list(group_mapping.values())}")

    return group_mapping



## Step 10: Extract Metadata from NetCDF and ClimateData.ca

Extract variable metadata from NetCDF attributes and supplement with information
from ClimateData.ca variables page.


In [ ]:
def extract_netcdf_metadata(zarr_store_path, cache_dir=None):
    """
    Extract metadata from zarr store attributes (inherited from NetCDF).

    Returns:
        dict: Metadata by variable name
    """
    print(f"\nExtracting metadata from zarr attributes...")

    metadata = {}

    # Open first available resolution group to get variable metadata
    # Try resolutions in descending order (highest resolution has most detail)
    zarr_root = zarr.open_group(zarr_store_path, mode='r')
    available_groups = sorted([g for g in zarr_root.group_keys()], reverse=True)

    if not available_groups:
        print(f"  ⚠️  No groups found in zarr store")
        return metadata

    first_group = available_groups[0]
    print(f"  Reading metadata from group: {first_group}")

    ds = xr.open_zarr(zarr_store_path, group=first_group, decode_timedelta=False)

    for var_name in ds.data_vars:
        var_attrs = ds[var_name].attrs

        metadata[var_name] = {
            "title": var_attrs.get("long_name", var_name),
            "description": var_attrs.get("description", ""),
            "x-ogc-unit": var_attrs.get("units", ""),
            "url": None  # Will be filled by web scraping
        }

        # Try to extract CF definition reference
        if "standard_name" in var_attrs:
            metadata[var_name]["x-ogc-definition"] = (
                f"http://cfconventions.org/Data/cf-standard-names/current/build/cf-standard-name-table.html#{var_attrs['standard_name']}"
            )

    ds.close()

    return metadata

def supplement_metadata_from_web(metadata, cache_dir=None):
    """
    Supplement metadata with information from climatedata.ca/variables/.
    Uses caching to avoid repeated web requests.
    """
    if cache_dir:
        cache_file = Path(cache_dir) / "climate_variable_metadata.json"

        if cache_file.exists():
            print(f"Loading metadata from cache: {cache_file}")
            with open(cache_file, mode="r", encoding="utf-8") as f:
                cached = json.load(f)
                for var_name in metadata:
                    if var_name in cached and cached[var_name].get("url"):
                        metadata[var_name]["url"] = cached[var_name]["url"]
                return metadata

    # Fetch variable slugs from ClimateData.ca
    try:
        url = "https://climatedata.ca/variables/"
        print(f"Fetching variable information from {url}...")
        resp = requests.get(url, timeout=15, allow_redirects=True)

        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            var_links = soup.find_all("a", href=lambda x: x and "/variable/" in x)

            slug_map = {}
            for link in var_links:
                href = link.get("href", "")
                if "/variable/" in href:
                    slug = href.split("/variable/")[-1].rstrip("/")
                    if slug:
                        full_url = href if href.startswith("http") else f"https://climatedata.ca{href}"
                        slug_map[slug] = full_url

            print(f"Found {len(slug_map)} variable slugs")

            # Match URLs to our variables
            # Variable names in NetCDF often have prefixes (ssp126_prcptot_p10)
            # We need to extract the core index name (prcptot)
            for var_name in metadata:
                # Try exact match first
                if var_name in slug_map:
                    metadata[var_name]["url"] = slug_map[var_name]
                    continue

                # Try hyphenated version
                hyphenated = var_name.replace("_", "-")
                if hyphenated in slug_map:
                    metadata[var_name]["url"] = slug_map[hyphenated]
                    continue

                # Extract core index name (find known climate indices)
                for known_idx in ["prcptot", "tx_max", "ice_days", "frost_days",
                                 "tn_min", "rx1day", "cdd", "growing_degree_days"]:
                    if known_idx in var_name.lower():
                        slug = known_idx.replace("_", "-")
                        if slug in slug_map:
                            metadata[var_name]["url"] = slug_map[slug]
                            break
        else:
            print(f"Warning: Could not fetch variables list (status {resp.status_code})")

    except Exception as e:
        print(f"Warning: Could not fetch variables listing: {e}")

    # Set fallback URLs for any missing
    for var_name in metadata:
        if not metadata[var_name].get("url"):
            # Try to extract core index name for fallback URL
            for known_idx in ["prcptot", "tx_max", "ice_days", "frost_days"]:
                if known_idx in var_name.lower():
                    metadata[var_name]["url"] = f"https://climatedata.ca/variable/{known_idx.replace('_', '-')}/"
                    break

    # Cache the metadata
    if cache_dir and cache_file:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_file, mode="w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)

    return metadata


## Step 11: Generate pydggsapi Configuration

Create the final configuration JSON for pydggsapi, including:
- DGGRS definition (H3)
- Collection metadata with proper links
- Collection provider with zone_groups mapping
- Data source configuration


In [ ]:
def compute_spatial_temporal_extent(zarr_path):
    """
    Compute spatial and temporal extent from the hierarchical zarr store.
    Uses the highest resolution data for spatial extent.
    """
    print(f"\nComputing spatial and temporal extent...")

    # Find highest resolution group
    zarr_root = zarr.open_group(zarr_path, mode='r')
    group_names = sorted([g for g in zarr_root.group_keys()],
                        key=lambda x: int(x.replace("res", "")),
                        reverse=True)

    if not group_names:
        raise ValueError(f"No resolution groups found in {zarr_path}")

    highest_res_group = group_names[0]
    print(f"  Using group '{highest_res_group}' for extent calculation")

    ds = xr.open_zarr(zarr_path, group=highest_res_group, decode_timedelta=False)

    # Get H3 cells and compute bounding box
    h3_ids = ds['h3_id'].values

    lats = []
    lons = []
    for h3_int in h3_ids[:1000]:  # Sample for performance
        h3_str = h3.int_to_str(h3_int)
        lat, lon = h3.cell_to_latlng(h3_str)
        lats.append(lat)
        lons.append(lon)

    spatial_bbox = [
        float(np.min(lons)),
        float(np.min(lats)),
        float(np.max(lons)),
        float(np.max(lats))
    ]

    # Get temporal extent
    time_values = pd.to_datetime(ds['time'].values)
    time_coords = sorted([t.isoformat() for t in time_values])

    temporal_range = {
        "interval": [[time_coords[0], time_coords[-1]]],
        "grid": {
            "cellsCount": len(time_coords),
            "coordinates": time_coords
        }
    }

    ds.close()

    return spatial_bbox, temporal_range

def build_pydggsapi_config(zarr_path, h3_resolution, variable_metadata,
                           spatial_bbox, temporal_range, cache_dir=None):
    """
    Build complete pydggsapi configuration JSON.
    """
    today = datetime.date.today().isoformat()

    # Get all data variable names from first available group
    zarr_root = zarr.open_group(zarr_path, mode='r')
    first_group = sorted([g for g in zarr_root.group_keys()])[0]
    ds = xr.open_zarr(zarr_path, group=first_group, decode_timedelta=False)
    data_cols = sorted(ds.data_vars)
    ds.close()

    # Build zone_groups mapping for all resolutions
    zone_groups = {str(i): f"res{i}" for i in range(h3_resolution + 1)}

    config = {
        "dggrs": {
            "1": {
                "H3": {
                    "title": "DGGRS H3",
                    "description": (
                        "H3 hexagonal hierarchical geospatial indexing system developed by Uber "
                        "(https://github.com/uber/h3, https://h3geo.org/)."
                    ),
                    "crs": "wgs84",
                    "shapeType": "hexagon",
                    "uri": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                    "definition_link": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                    "defaultDepth": 0,
                    "classname": "h3_dggrs_provider.H3Provider"
                }
            }
        },
        "collections": {
            "1": {
                "canada-climatedata": {
                    "title": "Canada Climate Indices DGGS Collection",
                    "description": (
                        "Climate indices mapped to H3 DGGS cells, aggregated from NetCDF sources "
                        "provided by ClimateData.ca."
                    ),
                    "attribution": f"ClimateData.ca [Accessed on {today}]",
                    "extent": {
                        "spatial": {
                            "bbox": [spatial_bbox],
                            "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"
                        },
                        "temporal": {
                            "interval": temporal_range["interval"],
                            "grid": temporal_range["grid"],
                            "trs": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian"
                        }
                    },
                    "links": [
                        {
                            "rel": "about",
                            "href": "https://climatedata.ca/",
                            "type": "text/html",
                            "title": "ClimateData.ca - Canadian Climate Data Portal",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "about",
                            "href": "https://donneesclimatiques.ca/",
                            "type": "text/html",
                            "title": "Portail canadien des données climatiques",
                            "hreflang": "fr-CA"
                        },
                        {
                            "rel": "license",
                            "href": "https://creativecommons.org/licenses/by/4.0/legalcode",
                            "type": "text/html",
                            "title": "Creative Commons Attribution International (CC BY)",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "terms-of-service",
                            "href": "https://climatedata.ca/about/legal/terms/",
                            "type": "text/html",
                            "title": "ClimateData.ca Terms of Service",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "describedby",
                            "href": "https://climatedata.ca/variables/",
                            "type": "text/html",
                            "title": "ClimateData.ca Variables Documentation",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "convertedfrom",
                            "href": "https://climatedata.ca/",
                            "type": "text/html",
                            "title": "Original ClimateData.ca Dataset Portal",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "via",
                            "href": "https://github.com/crim-ca/ogc-dggs/tree/main/canada-climate",
                            "type": "text/html",
                            "title": "Source code employed to generate DGGS data from reference features",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "cite-as",
                            "href": "https://hirondelle.crim.ca/dggs-api/collections/canada-climatedata",
                            "type": "application/json",
                            "title": "Canada Climate Indices as DGGS Collection",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "sponsored",
                            "href": "https://climatedata.ca/about/citing-climatedata-ca/",
                            "type": "text/html",
                            "title": "ClimateData.ca Sponsorship and Funding",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "author",
                            "href": "https://crim.ca/fr/",
                            "type": "text/html",
                            "hreflang": "fr-CA",
                            "title": "Centre de recherche informatique de Montréal (CRIM)"
                        },
                        {
                            "rel": "author",
                            "href": "https://crim.ca/en/",
                            "type": "text/html",
                            "hreflang": "en-CA",
                            "title": "Centre de recherche informatique de Montréal (CRIM)"
                        }
                    ],
                    "collection_provider": {
                        "providerId": "canada-climatedata",
                        "dggrsId": "H3",
                        "dggrs_zoneid_repr": "int",
                        "min_refinement_level": 0,
                        "max_refinement_level": h3_resolution,
                        "datasource_id": "canada-climatedata"
                    }
                }
            }
        },
        "collection_providers": {
            "1": {
                "canada-climatedata": {
                    "classname": "zarr_collection_provider.ZarrCollectionProvider",
                    "datasources": {
                        "canada-climatedata": {
                            "filepath": str(Path(zarr_path).absolute()),
                            "zone_groups": zone_groups,
                            "id_col": "h3_id",
                            "datetime_col": "time",
                            "data_cols": data_cols,
                            "exclude_data_cols": []
                        }
                    }
                }
            }
        }
    }

    return config


## Step 12: Execute Final Integration

Combine all individual zarr files into hierarchical zarr structure and generate pydggsapi config.


In [ ]:
if 'H3_RESOLUTION' in globals() and H3_RESOLUTION is not None:
    print(f"\n{'='*70}")
    print(f"FINAL INTEGRATION")
    print(f"{'='*70}\n")

    # Step 1: Combine zarr files into hierarchical structure
    print("Step 1: Creating hierarchical zarr structure...")
    group_mapping = combine_zarr_files_to_grouped_zarr(
        output_base_dir=OUTPUT_DIR,
        h3_resolution=H3_RESOLUTION,
        final_output_path=FINAL_ZARR_PATH
    )

    # Step 2: Extract and supplement metadata
    print("\nStep 2: Extracting metadata...")
    variable_metadata = extract_netcdf_metadata(FINAL_ZARR_PATH, cache_dir=CACHE_DIR)
    variable_metadata = supplement_metadata_from_web(variable_metadata, cache_dir=CACHE_DIR)

    print(f"\nExtracted metadata for {len(variable_metadata)} variables")

    # Step 3: Compute spatial and temporal extent
    print("\nStep 3: Computing extents...")
    spatial_bbox, temporal_range = compute_spatial_temporal_extent(FINAL_ZARR_PATH)

    print(f"Spatial bbox: {spatial_bbox}")
    print(f"Temporal range: {temporal_range['interval']}")
    print(f"Timesteps: {temporal_range['grid']['cellsCount']}")

    # Step 4: Build pydggsapi configuration
    print("\nStep 4: Generating pydggsapi configuration...")
    pydggsapi_config = build_pydggsapi_config(
        zarr_path=FINAL_ZARR_PATH,
        h3_resolution=H3_RESOLUTION,
        variable_metadata=variable_metadata,
        spatial_bbox=spatial_bbox,
        temporal_range=temporal_range,
        cache_dir=CACHE_DIR
    )

    # Save configuration
    with open(FINAL_CONFIG_PATH, 'w', encoding='utf-8') as f:
        json.dump(pydggsapi_config, f, indent=2)

    print(f"\n{'='*70}")
    print(f"✅ INTEGRATION COMPLETE")
    print(f"{'='*70}")
    print(f"Hierarchical zarr: {FINAL_ZARR_PATH}")
    print(f"Config JSON: {FINAL_CONFIG_PATH}")
    print(f"\nResolutions: 0 to {H3_RESOLUTION}")
    print(f"Resolution groups: {list(group_mapping.values())}")
    print(f"Variables: {len(pydggsapi_config['collection_providers']['1']['canada-climatedata']['datasources']['canada-climatedata']['data_cols'])}")

else:
    print("H3_RESOLUTION not defined. Run previous cells first.")
